# Teste de Segmentação — Notebook Standalone

Este notebook isola apenas a etapa de **segmentação de imagens** (Otsu +
maior componente conectado + convex hull preenchido) do pipeline principal
(`[VERSÃO 7] Modelo_Desvio_Hiperparametro`), para permitir testar e
visualizar a segmentação sem precisar rodar o treino do modelo.

A função `segment_image_cv` usada aqui é **idêntica** à usada no notebook
de treino (seção 1.5.1), garantindo que os testes reflitam fielmente o
pré-processamento real aplicado ao modelo.

## Estrutura do notebook
1. Configuração do ambiente e download do dataset
2. Função de segmentação
3. Teste visual (etapas da segmentação, algumas imagens por classe)
4. Teste estatístico (fração de pixels mantida, detecção de possíveis falhas)
5. Amostra final da segmentação para o artigo (1 imagem por classe)


## 1. Configuração do ambiente e download do dataset

In [ ]:
# --------------------------------------------------------------
# 1.1. (Opcional) Montar o Google Drive
#      Só necessário se você quiser salvar as figuras geradas neste
#      notebook no Drive. Para apenas visualizar em tela, pode pular.
# --------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --------------------------------------------------------------
# 1.2. Importações principais
# --------------------------------------------------------------
import os

import numpy as np
import cv2
import matplotlib.pyplot as plt
import kagglehub
import tensorflow as tf

# --------------------------------------------------------------
# 1.3. Download do dataset (Kaggle) e coleta dos caminhos das imagens
# --------------------------------------------------------------
path = kagglehub.dataset_download("mohammadhossein77/brain-tumors-dataset")
print("Caminho do dataset:", path)

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

image_paths = []
labels_raw = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            image_paths.append(os.path.join(root, file))
            labels_raw.append(os.path.basename(root))  # nome da pasta = classe

# --------------------------------------------------------------
# 1.4. Mapear nomes das classes
# --------------------------------------------------------------
class_names_list = sorted(list(set(labels_raw)))
num_classes = len(class_names_list)

image_paths = np.array(image_paths)
image_paths_originais = image_paths  # nome mantido por compatibilidade com as funções de teste abaixo

print(f"\nTotal de imagens encontradas: {len(image_paths)}")
print(f"Classes identificadas: {class_names_list}")
for classe in class_names_list:
    count = labels_raw.count(classe)
    print(f" - {classe}: {count} imagens")


## 2. Função de segmentação

Otsu (separa objeto do fundo por intensidade) + maior componente
conectado (remove ruído solto) + convex hull preenchido. O convex hull é
o que resolve o principal problema do Otsu puro: como o Otsu decide
pixel a pixel só pela intensidade, um tumor com brilho parecido com o
fundo podia "sumir" junto (virar buraco na máscara). Preenchendo o fecho
convexo do contorno externo do cérebro, garante-se que nada DENTRO do
contorno seja excluído, independente da intensidade interna.


In [ ]:
IMAGE_SIZE = (224, 224)

def compute_mask(img):
    """
    Calcula a máscara final de segmentação para uma imagem já redimensionada
    (uint8, RGB): Otsu -> maior componente conectado -> convex hull preenchido.
    Isola essa lógica para ser reaproveitada tanto por `segment_image_cv`
    (usada no treino) quanto pelos testes abaixo, garantindo que todos
    usem EXATAMENTE a mesma máscara.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)

    # Threshold automático (Otsu) para separar objeto do fundo
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Mantém só o maior componente conectado (remove ruído solto do fundo)
    n_labels, labels_img, stats, _ = cv2.connectedComponentsWithStats(thresh, connectivity=8)
    if n_labels > 1:  # label 0 é sempre o fundo
        maior_componente = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        thresh = np.where(labels_img == maior_componente, 255, 0).astype(np.uint8)

    # Preenche o fecho convexo do contorno externo — garante que nada
    # dentro do cérebro (ex.: tumor com intensidade parecida ao fundo)
    # seja cortado pela máscara.
    contornos, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contornos:
        maior_contorno = max(contornos, key=cv2.contourArea)
        hull = cv2.convexHull(maior_contorno)
        thresh_hull = np.zeros_like(thresh)
        cv2.drawContours(thresh_hull, [hull], -1, 255, thickness=cv2.FILLED)
        thresh = thresh_hull

    return thresh

def segment_image_cv(image_tensor):
    """
    Função de segmentação EXATAMENTE igual à usada no notebook de treino
    (seção 1.5.1): Otsu + maior componente conectado + convex hull preenchido.
    """
    img = image_tensor.numpy().astype(np.uint8) if hasattr(image_tensor, 'numpy') else image_tensor.astype(np.uint8)

    # Se a imagem vier com shape (1, 224, 224, 3), remove a dimensão extra
    if len(img.shape) == 4:
        img = np.squeeze(img, axis=0)

    mask = compute_mask(img)

    # Aplica a máscara resultante sobre a imagem original
    segmented = cv2.bitwise_and(img, img, mask=mask)
    return segmented.astype(np.float32)


## 3. Teste visual: etapas da segmentação

Para algumas imagens de cada classe, mostra lado a lado: original ->
escala de cinza -> máscara do Otsu -> imagem segmentada.


In [ ]:
def testar_segmentacao_visual(n_por_classe=2, seed=42):
    """
    Para algumas imagens de cada classe, mostra lado a lado:
    original -> escala de cinza -> máscara final (Otsu + hull) -> imagem segmentada.
    Usa exatamente a mesma máscara (`compute_mask`) que `segment_image_cv`
    aplica no treino.
    """
    rng = np.random.default_rng(seed)
    amostras = []

    for classe in class_names_list:
        indices_classe = [i for i, lbl in enumerate(labels_raw) if lbl == classe]
        n = min(n_por_classe, len(indices_classe))
        escolhidos = rng.choice(indices_classe, size=n, replace=False)
        for idx in escolhidos:
            amostras.append((classe, image_paths_originais[idx]))

    n_linhas = len(amostras)
    fig, axes = plt.subplots(n_linhas, 4, figsize=(14, 3.2 * n_linhas))
    if n_linhas == 1:
        axes = axes.reshape(1, -1)

    titulos = ["Original", "Escala de cinza", "Máscara (Otsu + hull)", "Segmentado"]

    for row, (classe, caminho) in enumerate(amostras):
        img_raw = tf.io.read_file(caminho)
        img_raw = tf.image.decode_jpeg(img_raw, channels=3)
        img_resized = tf.image.resize(img_raw, IMAGE_SIZE).numpy().astype(np.uint8)

        gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
        mask = compute_mask(img_resized)
        segmentado = cv2.bitwise_and(img_resized, img_resized, mask=mask)

        imagens = [img_resized, gray, mask, segmentado]
        cmaps = [None, 'gray', 'gray', None]

        for col, (imagem, cmap) in enumerate(zip(imagens, cmaps)):
            axes[row, col].imshow(imagem, cmap=cmap)
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(titulos[col])
        axes[row, 0].set_ylabel(classe)

    plt.tight_layout()
    plt.show()

testar_segmentacao_visual(n_por_classe=2)


## 4. Teste estatístico: varre uma amostra maior procurando falhas

Roda a segmentação em uma amostra aleatória de imagens e calcula a
fração de pixels mantida pela máscara em cada uma, sinalizando imagens
fora da faixa esperada (possíveis falhas de segmentação).


In [ ]:
def testar_segmentacao_estatistico(n_amostras=200, seed=42,
                                    limite_baixo=0.02, limite_alto=0.90):
    """
    Roda a segmentação (máscara final, com convex hull) em uma amostra
    aleatória de imagens e calcula a fração de pixels mantida em cada uma.
    """
    rng = np.random.default_rng(seed)
    n_amostras = min(n_amostras, len(image_paths_originais))
    indices = rng.choice(len(image_paths_originais), size=n_amostras, replace=False)

    fracoes = []
    suspeitas = []

    for idx in indices:
        caminho = image_paths_originais[idx]
        classe = labels_raw[idx]

        img_raw = tf.io.read_file(caminho)
        img_raw = tf.image.decode_jpeg(img_raw, channels=3)
        img_resized = tf.image.resize(img_raw, IMAGE_SIZE).numpy().astype(np.uint8)

        mask = compute_mask(img_resized)

        fracao = (mask > 0).mean()
        fracoes.append(fracao)

        if fracao < limite_baixo or fracao > limite_alto:
            suspeitas.append((caminho, classe, fracao))

    fracoes = np.array(fracoes)
    print(f"Amostra: {n_amostras} imagens")
    print(f"Fração de pixels mantida pela máscara — média: {fracoes.mean():.3f} "
          f"| mín: {fracoes.min():.3f} | máx: {fracoes.max():.3f}")
    print(f"Imagens suspeitas (fora de [{limite_baixo}, {limite_alto}]): "
          f"{len(suspeitas)} de {n_amostras} ({100 * len(suspeitas) / n_amostras:.1f}%)")

    if suspeitas:
        print("\nPrimeiras suspeitas encontradas:")
        for caminho, classe, fracao in suspeitas[:10]:
            print(f" - [{classe}] fração={fracao:.3f} | {caminho}")

    return fracoes, suspeitas

fracoes, suspeitas = testar_segmentacao_estatistico(n_amostras=200)


## 5. Amostra final da segmentação para o artigo

Gera 1 imagem por classe (original, máscara, segmentado) usando a função oficial `segment_image_cv`.

In [ ]:
# --------------------------------------------------------------
# Amostra de segmentação: 1 imagem por classe (original, máscara, segmentado)
# --------------------------------------------------------------
def amostra_segmentacao_por_classe(seed=42):
    rng = np.random.default_rng(seed)
    amostras = []

    for classe in class_names_list:
        indices_classe = [i for i, lbl in enumerate(labels_raw) if lbl == classe]
        idx = rng.choice(indices_classe)
        amostras.append((classe, image_paths_originais[idx]))

    fig, axes = plt.subplots(len(amostras), 3, figsize=(10, 3.2 * len(amostras)))

    titulos = ["Original", "Máscara", "Segmentado"]

    for row, (classe, caminho) in enumerate(amostras):
        img_raw = tf.io.read_file(caminho)
        img_raw = tf.image.decode_jpeg(img_raw, channels=3)
        img_resized = tf.image.resize(img_raw, IMAGE_SIZE).numpy().astype(np.uint8)

        mascara = compute_mask(img_resized)
        segmentado = segment_image_cv(img_resized).astype(np.uint8)

        imagens = [img_resized, mascara, segmentado]
        cmaps = [None, 'gray', None]

        for col, (imagem, cmap) in enumerate(zip(imagens, cmaps)):
            axes[row, col].imshow(imagem, cmap=cmap)
            axes[row, col].axis('off')
            if row == 0:
                axes[row, col].set_title(titulos[col])
        axes[row, 0].set_ylabel(classe, rotation=90, labelpad=10)

    plt.tight_layout()
    plt.show()
    return fig

fig = amostra_segmentacao_por_classe()
